# Notebook 2: Modélisation

## Credit Scoring avec IA Explicable (XAI)

### Objectifs
- Entraîner un modèle baseline (Logistic Regression)
- Entraîner des modèles XGBoost et LightGBM
- Optimiser les hyperparamètres
- Comparer les performances des modèles
- Sélectionner le meilleur modèle

In [ ]:
# Import des bibliothèques
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import des modules du projet
from src.data_loader import load_data
from src.preprocessing import DataPreprocessor, split_data
from src.models.baseline_model import BaselineModel
from src.models.xgboost_model import XGBoostModel
from src.models.lightgbm_model import LightGBMModel
from src.evaluation import ModelEvaluator
from src.config import MODELS_DIR

print("✅ Import des bibliothèques réussi")

## 1. Chargement et Prétraitement des Données

In [ ]:
# Charger les données
X, y = load_data()

print(f"✅ Données chargées: {X.shape}")
print(f"   - Features: {X.shape[1]}")
print(f"   - Instances: {X.shape[0]}")

In [ ]:
# Prétraitement
preprocessor = DataPreprocessor()
X_transformed = preprocessor.fit_transform(X)

print(f"✅ Données prétraitées: {X_transformed.shape}")
print(f"   - Features numériques: {len(preprocessor.feature_names)}")

In [ ]:
# Split des données
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X_transformed, y)

print(f"✅ Données divisées:")
print(f"   - Train: {X_train.shape[0]} instances")
print(f"   - Validation: {X_val.shape[0]} instances")
print(f"   - Test: {X_test.shape[0]} instances")

## 2. Modèle Baseline: Logistic Regression

In [ ]:
# Créer et entraîner le modèle baseline
baseline = BaselineModel()
baseline_metrics = baseline.train(X_train, y_train, X_val, y_val)

print("\n📊 Métriques d'entraînement (Baseline - Logistic Regression):")
for key, value in baseline_metrics.items():
    print(f"   - {key}: {value:.4f}")

In [ ]:
# Évaluer sur le test set
baseline_test_metrics = baseline.evaluate(X_test, y_test)

print("\n📊 Métriques de test (Baseline):")
for key, value in baseline_test_metrics.items():
    print(f"   - {key}: {value:.4f}")

In [ ]:
# Importance des features (coefficients absolus)
baseline_importance = baseline.get_feature_importance()

print("\n📊 Top 10 features (Baseline):")
print(baseline_importance.head(10))

In [ ]:
# Sauvegarder le modèle
baseline.save(str(MODELS_DIR / "baseline_model.pkl"))
print("\n✅ Modèle baseline sauvegardé")

## 3. Modèle XGBoost

In [ ]:
# Créer et entraîner le modèle XGBoost
xgb = XGBoostModel()
xgb_metrics = xgb.train(X_train, y_train, X_val, y_val)

print("\n📊 Métriques d'entraînement (XGBoost):")
for key, value in xgb_metrics.items():
    print(f"   - {key}: {value:.4f}")

In [ ]:
# Évaluer sur le test set
xgb_test_metrics = xgb.evaluate(X_test, y_test)

print("\n📊 Métriques de test (XGBoost):")
for key, value in xgb_test_metrics.items():
    print(f"   - {key}: {value:.4f}")

In [ ]:
# Importance des features (gain)
xgb_importance = xgb.get_feature_importance(importance_type='gain')

print("\n📊 Top 10 features (XGBoost - Gain):")
print(xgb_importance.head(10))

In [ ]:
# Sauvegarder le modèle
xgb.save(str(MODELS_DIR / "xgboost_model.pkl"))
print("\n✅ Modèle XGBoost sauvegardé")

## 4. Modèle LightGBM

In [ ]:
# Créer et entraîner le modèle LightGBM
lgb = LightGBMModel()
lgb_metrics = lgb.train(X_train, y_train, X_val, y_val)

print("\n📊 Métriques d'entraînement (LightGBM):")
for key, value in lgb_metrics.items():
    print(f"   - {key}: {value:.4f}")

In [ ]:
# Évaluer sur le test set
lgb_test_metrics = lgb.evaluate(X_test, y_test)

print("\n📊 Métriques de test (LightGBM):")
for key, value in lgb_test_metrics.items():
    print(f"   - {key}: {value:.4f}")

In [ ]:
# Importance des features (split)
lgb_importance = lgb.get_feature_importance(importance_type='split')

print("\n📊 Top 10 features (LightGBM - Split):")
print(lgb_importance.head(10))

In [ ]:
# Sauvegarder le modèle
lgb.save(str(MODELS_DIR / "lightgbm_model.pkl"))
print("\n✅ Modèle LightGBM sauvegardé")

## 5. Comparaison des Modèles

In [ ]:
# Créer l'évaluateur
evaluator = ModelEvaluator()

# Comparer les modèles
models = {
    'Baseline (Logistic Regression)': baseline,
    'XGBoost': xgb,
    'LightGBM': lgb
}

comparison = evaluator.compare_models(models, X_test, y_test)

print("\n📊 Comparaison des modèles:")
print(comparison.to_string(index=False))

In [ ]:
# Visualiser la comparaison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Avg Precision']

for i, metric in enumerate(metrics):
    sns.barplot(data=comparison, x='Model', y=metric, ax=axes[i])
    axes[i].set_title(f'{metric} par Modèle')
    axes[i].set_xlabel('Modèle')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=45)

# Supprimer l'axe vide
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()

In [ ]:
# Courbes ROC comparatives
fig = evaluator.plot_comparison_roc(models, X_test, y_test)
plt.show()

In [ ]:
# Meilleur modèle
best_model_name, best_model_metrics = evaluator.get_best_model('roc_auc')

print(f"\n🏆 Meilleur modèle: {best_model_name}")
print(f"   ROC-AUC: {best_model_metrics['roc_auc']:.4f}")
print(f"   F1-Score: {best_model_metrics['f1']:.4f}")
print(f"   Accuracy: {best_model_metrics['accuracy']:.4f}")

## 6. Optimisation des Hyperparamètres (Optionnel)

In [ ]:
# Optimisation des hyperparamètres XGBoost (Random Search)
# Note: Cette étape peut prendre plusieurs minutes

xgb_opt = XGBoostModel()
xgb_opt_metrics = xgb_opt.hyperparameter_tuning(
    X_train, y_train,
    X_val, y_val,
    method='random',
    n_iter=50,
    cv=3
)

print("\n📊 Métriques après optimisation (XGBoost):")
for key, value in xgb_opt_metrics.items():
    print(f"   - {key}: {value:.4f}")

In [ ]:
# Sauvegarder le modèle optimisé
xgb_opt.save(str(MODELS_DIR / "xgboost_model_optimized.pkl"))
print("\n✅ Modèle XGBoost optimisé sauvegardé")

## 7. Résumé

In [ ]:
print("\n" + "="*60)
print("📊 RÉSUMÉ DE LA MODÉLISATION")
print("="*60)

print("\n📋 Modèles entraînés:")
print(f"   1. Baseline (Logistic Regression) - ROC-AUC: {baseline_test_metrics['roc_auc']:.4f}")
print(f"   2. XGBoost - ROC-AUC: {xgb_test_metrics['roc_auc']:.4f}")
print(f"   3. LightGBM - ROC-AUC: {lgb_test_metrics['roc_auc']:.4f}")

print(f"\n🏆 Meilleur modèle: {best_model_name}")
print(f"   - ROC-AUC: {best_model_metrics['roc_auc']:.4f}")
print(f"   - F1-Score: {best_model_metrics['f1']:.4f}")
print(f"   - Accuracy: {best_model_metrics['accuracy']:.4f}")

print("\n📊 Top 5 features (LightGBM):")
print(lgb_importance.head(5))

print("\n" + "="*60)
print("✅ Modélisation terminée avec succès")
print("="*60)